### Cell 1 Configuration

In [12]:
import os
import torch

class Config:
    CSV_TRAIN = "train.csv"
    IMG_DIR_TRAIN = "data/train"
    
    # Model Settings
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  
    
    # Hyperparameters
    EPOCHS = 15
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    VAL_SPLIT_RATIO = 0.2
    RANDOM_SEED = 42
    
    # Hardware Device Detection
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {Config.DEVICE}")
if Config.DEVICE.type == 'cuda':
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

Using Device: cuda
🔥 GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Dataset and DataLoader

In [13]:
# CELL 2: DATASET & DATALOADERS (Using ImageFolder)

import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from dataset import IlluminationDataset, train_transforms, val_transforms

In [14]:
# 1. Load the complete training CSV
full_df = pd.read_csv(Config.CSV_TRAIN)

# 2. Split data into 80% training and 20% validation
train_df, val_df = train_test_split(
    full_df,
    test_size=Config.VAL_SPLIT_RATIO,
    random_state=Config.RANDOM_SEED,
    stratify=full_df["label"]
)

# 3. Create training and validation datasets
train_dataset = IlluminationDataset(
    dataframe=train_df,
    img_dir=Config.IMG_DIR_TRAIN,
    transform=train_transforms
)

val_dataset = IlluminationDataset(
    dataframe=val_df,
    img_dir=Config.IMG_DIR_TRAIN,
    transform=val_transforms
)

# 4. Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

# 5. Verify the split
print(f"Total labeled images: {len(full_df)}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["label"].value_counts().sort_index())

Total labeled images: 1500
Training images: 1200
Validation images: 300

Training class distribution:
label
0    400
1    400
2    400
Name: count, dtype: int64

Validation class distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64


### CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

In [15]:
# CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

import torch.nn as nn
import torchvision.models as models

def build_illumination_model():
    # 1. Load pretrained ResNet18
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # 2. Freeze all pretrained layers
    for param in model.parameters():
        param.requires_grad = False

    # 3. Unfreeze only the final residual block
    for param in model.layer4[1].parameters():
        param.requires_grad = True

    # 4. Replace the classification head
    num_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_features, 3)
    )

    return model


model = build_illumination_model().to(Config.DEVICE)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Model Initialized on {Config.DEVICE}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Model Initialized on cuda
Total Parameters: 11,178,051
Trainable Parameters: 4,722,179


#### Training on 8 million parameters

### CELL 4: TRAINING & VALIDATION ENGINE

In [16]:
from tqdm import tqdm # Great for visual progress bars in VS Code

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # Progress bar for the terminal/notebook
    loop = tqdm(dataloader, leave=False, desc="Training")
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        # 1. Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward Pass & Optimization
        
        loss.backward()
        optimizer.step()
        
        # 3. Track Metrics
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1) # Get the index of the highest logit
        correct_preds += torch.sum(preds == labels).item()
        total_samples += labels.size(0)
        
        # Update progress bar
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    with torch.no_grad(): # Disable gradient tracking for speed and memory
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

             # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Track metrics
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_preds += torch.sum(preds == labels).item()
            total_samples += labels.size(0)
            
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

In [17]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([1, 0, 1, 2, 2, 0, 0, 2, 2, 2])


### CELL 5: MAIN EXECUTION LOOP

In [18]:
import time
# 1. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

# ONLY pass the trainable parameters to the optimizer
optimizer = torch.optim.AdamW(
    [
        {
            'params': model.layer4[1].parameters(),
            'lr': 5e-5
        },
        {
            'params': model.fc.parameters(),
            'lr': 2e-4
        }
    ],
    weight_decay=Config.WEIGHT_DECAY
)
# 1.5. Define Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

# 2. Initialize Tracking Variables
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🔥 Starting Training...")
start_time = time.time()

# 3. The Main Epoch Loop
for epoch in range(Config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
    print("-" * 20)
    
    # Train and Validate
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, Config.DEVICE)

    # Update learning rate based on validation loss
    scheduler.step(val_loss)
    
    # store metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Get current learning rate
    backbone_lr = optimizer.param_groups[0]['lr']
    fc_lr = optimizer.param_groups[1]['lr']

    print(f"Backbone LR: {backbone_lr:.8f}")
    print(f"FC LR:       {fc_lr:.8f}")
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
   
    
    # 4. Checkpointing Strategy (Save ONLY if Accuracy improves)
    if val_acc > best_val_acc:
        print(f"⭐ Validation Accuracy improved from {best_val_acc:.4f} to {val_acc:.4f}. Saving checkpoint!")

        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

# 5. Calculate total training time
time_elapsed = time.time() - start_time
print(
    f"\n✅ Training Complete in "
    f"{time_elapsed // 60:.0f}m "
    f"{time_elapsed % 60:.0f}s"
)
print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")

🔥 Starting Training...

Epoch 1/15
--------------------


Backbone LR: 0.00005000
FC LR:       0.00020000
Train Loss: 1.1621 | Train Acc: 0.3792
Val Loss:   1.1174 | Val Acc:   0.4133
⭐ Validation Accuracy improved from 0.0000 to 0.4133. Saving checkpoint!

Epoch 2/15
--------------------


Backbone LR: 0.00005000
FC LR:       0.00020000
Train Loss: 1.0895 | Train Acc: 0.4633
Val Loss:   1.0916 | Val Acc:   0.4200
⭐ Validation Accuracy improved from 0.4133 to 0.4200. Saving checkpoint!

Epoch 3/15
--------------------


Backbone LR: 0.00005000
FC LR:       0.00020000
Train Loss: 1.0710 | Train Acc: 0.4783
Val Loss:   1.0699 | Val Acc:   0.4367
⭐ Validation Accuracy improved from 0.4200 to 0.4367. Saving checkpoint!

Epoch 4/15
--------------------


Backbone LR: 0.00005000
FC LR:       0.00020000
Train Loss: 1.0147 | Train Acc: 0.5092
Val Loss:   1.0719 | Val Acc:   0.4667
⭐ Validation Accuracy improved from 0.4367 to 0.4667. Saving checkpoint!

Epoch 5/15
--------------------


...

Backbone LR: 0.00001250
FC LR:       0.00005000
Train Loss: 0.9065 | Train Acc: 0.6100
Val Loss:   1.0799 | Val Acc:   0.4800

Epoch 12/15
--------------------


Backbone LR: 0.00000625
FC LR:       0.00002500
Train Loss: 0.8571 | Train Acc: 0.6600
Val Loss:   1.0805 | Val Acc:   0.4900

Epoch 13/15
--------------------


Backbone LR: 0.00000625
FC LR:       0.00002500
Train Loss: 0.8776 | Train Acc: 0.6142
Val Loss:   1.0862 | Val Acc:   0.5000

Epoch 14/15
--------------------


Backbone LR: 0.00000625
FC LR:       0.00002500
Train Loss: 0.8766 | Train Acc: 0.6300
Val Loss:   1.0803 | Val Acc:   0.4933

Epoch 15/15
--------------------


Backbone LR: 0.00000313
FC LR:       0.00001250
Train Loss: 0.8421 | Train Acc: 0.6542
Val Loss:   1.0789 | Val Acc:   0.4867

✅ Training Complete in 2m 38s
🏆 Best Validation Accuracy: 0.5000


In [19]:
for epoch in range(len(history['train_loss'])):
    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Acc: {history['train_acc'][epoch]:.4f} | "
        f"Val Acc: {history['val_acc'][epoch]:.4f} | "
        f"Train Loss: {history['train_loss'][epoch]:.4f} | "
        f"Val Loss: {history['val_loss'][epoch]:.4f}"
    )

Epoch 01 | Train Acc: 0.3792 | Val Acc: 0.4133 | Train Loss: 1.1621 | Val Loss: 1.1174
Epoch 02 | Train Acc: 0.4633 | Val Acc: 0.4200 | Train Loss: 1.0895 | Val Loss: 1.0916
Epoch 03 | Train Acc: 0.4783 | Val Acc: 0.4367 | Train Loss: 1.0710 | Val Loss: 1.0699
Epoch 04 | Train Acc: 0.5092 | Val Acc: 0.4667 | Train Loss: 1.0147 | Val Loss: 1.0719
Epoch 05 | Train Acc: 0.5467 | Val Acc: 0.4900 | Train Loss: 0.9954 | Val Loss: 1.0782
Epoch 06 | Train Acc: 0.5608 | Val Acc: 0.4567 | Train Loss: 0.9657 | Val Loss: 1.0886
Epoch 07 | Train Acc: 0.5983 | Val Acc: 0.4700 | Train Loss: 0.9130 | Val Loss: 1.0777
Epoch 08 | Train Acc: 0.6050 | Val Acc: 0.4800 | Train Loss: 0.9195 | Val Loss: 1.0837
Epoch 09 | Train Acc: 0.5958 | Val Acc: 0.5000 | Train Loss: 0.9127 | Val Loss: 1.0753
Epoch 10 | Train Acc: 0.6050 | Val Acc: 0.4900 | Train Loss: 0.8994 | Val Loss: 1.0775
Epoch 11 | Train Acc: 0.6100 | Val Acc: 0.4800 | Train Loss: 0.9065 | Val Loss: 1.0799
Epoch 12 | Train Acc: 0.6600 | Val Acc: 0.4

In [20]:
model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=Config.DEVICE,
        weights_only=True
    )
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [21]:
from sklearn.metrics import confusion_matrix, classification_report

all_labels = []
all_preds = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(Config.DEVICE)
        labels = labels.to(Config.DEVICE)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["dark", "normal", "bright"]
    )
)

Confusion Matrix:
[[70 16 14]
 [30 53 17]
 [36 37 27]]

Classification Report:
              precision    recall  f1-score   support

        dark       0.51      0.70      0.59       100
      normal       0.50      0.53      0.51       100
      bright       0.47      0.27      0.34       100

    accuracy                           0.50       300
   macro avg       0.49      0.50      0.48       300
weighted avg       0.49      0.50      0.48       300



In [22]:
print("Train DataFrame:", len(train_df))
print("Validation DataFrame:", len(val_df))
print("Train Dataset:", len(train_dataset))
print("Validation Dataset:", len(val_dataset))

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

Train DataFrame: 1200
Validation DataFrame: 300
Train Dataset: 1200
Validation Dataset: 300

Validation distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64
